# Fabric Defect Detection — DINOv2 + PatchCore (Colab)

Unsupervised: fit on **defect-free** fabric only, flag deviations. Produces
image AUROC / pixel AUROC / PRO plus heatmap + bounding-box overlays.

**Runtime → Change runtime type → GPU (T4)** before running.

## 1. Setup — clone repo (or upload the project) & install deps

In [ ]:
# Option A: if you pushed this project to GitHub, clone it:
# !git clone https://github.com/<you>/fabric_defect.git
# %cd fabric_defect

# Option B: mount Drive and cd into your uploaded copy:
from google.colab import drive
drive.mount('/content/drive')
# %cd /content/drive/MyDrive/fabric_defect

!pip -q install torch torchvision scikit-learn scipy opencv-python Pillow tqdm

## 2. Get MVTec AD
Download once to Drive so you don't re-download every session.

In [ ]:
import os
DATA_ROOT = '/content/mvtec'   # or a Drive path to persist it
# The MVTec AD tarball requires accepting their terms; download the link from
# https://www.mvtec.com/company/research/datasets/mvtec-ad and extract so you get
#   {DATA_ROOT}/carpet/train/good/*.png
# Example once you have the tarball on Drive:
# !mkdir -p {DATA_ROOT} && tar -xf /content/drive/MyDrive/mvtec_anomaly_detection.tar.xz -C {DATA_ROOT}
assert os.path.exists(f'{DATA_ROOT}/carpet/train/good'), 'extract MVTec first'

## 3. Sanity check DINOv2 loads (downloads weights first time)

In [ ]:
import sys; sys.path.insert(0, '.')
import torch
from src.models.backbones import build_backbone
bb = build_backbone('dinov2_vits14')
x = torch.randn(2,3,224,224)
print('feature grid:', bb(x).shape)   # -> (2, 384, 16, 16)

## 4. Fit + evaluate one category
Saves `checkpoints/carpet.pt` and overlays to `outputs/`.

In [ ]:
!python scripts/train.py --data-root {DATA_ROOT} --category carpet \
    --model dinov2_vits14 --coreset 0.1 --image-size 224

## 5. View the heatmap + bounding-box overlays

In [ ]:
from IPython.display import Image as IPImage, display
import glob
for p in sorted(glob.glob('outputs/carpet_*.png'))[:4]:
    print(p); display(IPImage(p))

## 6. Paper experiment — DINOv2 vs WideResNet (CNN baseline)
Run both backbones across textile categories and collect the numbers.

In [ ]:
import subprocess, re
def run(model, cat):
    out = subprocess.run(['python','scripts/train.py','--data-root',DATA_ROOT,
        '--category',cat,'--model',model,'--coreset','0.1'],
        capture_output=True, text=True).stdout
    g = lambda k: float(re.search(k+r'\s*:\s*([0-9.]+)', out).group(1))
    return g('image AUROC'), g('pixel AUROC'), g('PRO')

rows = []
for cat in ['carpet','leather','grid']:
    for model in ['dinov2_vits14','wide_resnet50_2']:
        i,p,pro = run(model, cat)
        rows.append((cat, model, i, p, pro))
        print(f'{cat:8s} {model:16s} img={i:.3f} pix={p:.3f} pro={pro:.3f}')

In [ ]:
import pandas as pd
df = pd.DataFrame(rows, columns=['category','backbone','image_auroc','pixel_auroc','pro'])
df.to_csv('outputs/results.csv', index=False)
df

## 7. Save checkpoints to Drive (survive disconnects)

In [ ]:
!mkdir -p /content/drive/MyDrive/fabric_defect_ckpts
!cp checkpoints/*.pt /content/drive/MyDrive/fabric_defect_ckpts/ 2>/dev/null; echo done